<a href="https://colab.research.google.com/github/22417010/Gemini-Intelligence-Benchmarking/blob/main/Gemini%20Intelligence%20Benchmarking%20%E2%94%82%E2%94%82%20%5BRAG%20vs.%20Long%20Context%20System%5D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ============================================================
# Project: Gemini Intelligence Benchmarking (RAG vs. Long Context)
# Developer: Gemini Adaptive Collaborator
# Platform: Google Colab / OpenRouter API
# ============================================================



# 1. تثبيت المكتبات المطلوبة

In [ ]:

!pip install -q openai pypdf faiss-cpu sentence-transformers rich

import os
import time
import numpy as np
import faiss
from google.colab import files
from openai import OpenAI
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from rich.console import Console
from rich.panel import Panel
from rich.table import Table

console = Console()

# 2. الإعدادات (Configuration)
# تم استخدام مفتاح OpenRouter الخاص بك لضمان العمل بدون دفع

In [ ]:
import getpass
from google.colab import userdata

# 1. محاولة جلب المفتاح من نظام Secrets (إذا كان المستخدم هو أنت)
# 2. إذا لم يجد المفتاح، يظهر صندوق إدخال للمستخدم الزائر
try:
    OS_API_KEY = userdata.get('OPENROUTER_API_KEY')
except Exception:
    OS_API_KEY = None

if not OS_API_KEY:
    # هذا السطر هو الذي سيظهر للمستخدم في الديمو
    OS_API_KEY = getpass.getpass("الرجاء إدخال مفتاح OpenRouter الخاص بك لتشغيل الديمو: ")

# إعداد العميل باستخدام المفتاح (سواء كان مخزناً أو تم إدخاله)
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OS_API_KEY,
)

# أسماء النماذج (تأكد من استخدام النسخة المجانية أو المتاحة)
MODEL_RAG = "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free"
MODEL_LONG = "google/gemini-2.0-flash-001"



# 3. محرك معالجة الوثائق (Document Processing Engine)

In [ ]:

class GeminiRAGEngine:
    def __init__(self):
        # استخدام موديل محلي ومجاني لتحويل النصوص إلى متجهات
        self.embed_model = SentenceTransformer('all-MiniLM-L6-v2')
        self.index = None
        self.chunks = []
        self.full_text = ""

    def process_pdf(self, pdf_path):
        """قراءة الـ PDF وتقسيمه هندسياً"""
        reader = PdfReader(pdf_path)
        text = ""
        for page in reader.pages:
            text += page.extract_text() + "\n"
        self.full_text = text

        # تقسيم النص: Chunk size 800 with 100 overlap
        self.chunks = [text[i:i + 800] for i in range(0, len(text), 700)]

        # بناء فهرس FAISS للبحث السريع
        embeddings = self.embed_model.encode(self.chunks, normalize_embeddings=True)
        dimension = embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dimension)
        self.index.add(embeddings.astype(np.float32))
        return len(self.chunks)

    def retrieve(self, query, k=3):
        """استرجاع أفضل القطع النصية المشابهة للسؤال"""
        query_emb = self.embed_model.encode([query], normalize_embeddings=True).astype(np.float32)
        _, indices = self.index.search(query_emb, k)
        return [self.chunks[i] for i in indices[0]]


# 4. وظيفة المقارنة المعيارية (Benchmarking Logic)


In [ ]:
def run_benchmark(engine, query):
    # --- المسار الأول: RAG Approach ---
    start_rag = time.time()
    context_chunks = engine.retrieve(query)
    context_text = "\n---\n".join(context_chunks)

    rag_prompt = f"Answer based ONLY on the context:\n{context_text}\nQuestion: {query}"
    rag_res = client.chat.completions.create(
        model=MODEL_RAG,
        messages=[{"role": "user", "content": rag_prompt}]
    )
    time_rag = time.time() - start_rag

    # --- المسار الثاني: Long Context Approach ---
    start_long = time.time()
    long_prompt = f"Full Document Context:\n{engine.full_text}\nQuestion: {query}"
    long_res = client.chat.completions.create(
        model=MODEL_LONG,
        messages=[{"role": "user", "content": long_prompt}]
    )
    time_long = time.time() - start_long

    return {
        "rag": {"ans": rag_res.choices[0].message.content, "time": time_rag},
        "long": {"ans": long_res.choices[0].message.content, "time": time_long}
    }


# 5. واجهة التشغيل النهائية (Main Interface)


In [ ]:

def main():
    console.print(Panel.fit("🧪 Gemini Intelligence Benchmarking\n[RAG vs. Long Context System]", style="bold cyan"))

    # رفع الملف
    uploaded = files.upload()
    if not uploaded: return
    file_path = list(uploaded.keys())[0]

    # المعالجة
    engine = GeminiRAGEngine()
    with console.status("[bold yellow]جاري بناء الفهرس الدلالي..."):
        num_chunks = engine.process_pdf(file_path)
    console.print(f"[bold green]✓ تم فهرسة المستند بنجاح ({num_chunks} قطعة نصية).[/bold green]")

    while True:
        raw_input = console.input("\n[bold magenta]أدخل سؤالك (أو 'exit' للخروج): [/bold magenta]")
        query = str(raw_input) # حل مشكلة AttributeError

        if query.lower() in ['exit', 'خروج', 'quit']: break

        with console.status("[bold green]جاري تشغيل المقارنة المعيارية..."):
            results = run_benchmark(engine, query)

        # عرض النتائج في جدول احترافي
        table = Table(title="📊 المقارنة المعيارية للنتائج")
        table.add_column("الخاصية (Metric)", style="cyan")
        table.add_column("نهج الـ RAG (المحسن)", style="green")
        table.add_column("نهج السياق الطويل (الخام)", style="blue")

        table.add_row("زمن الاستجابة", f"{results['rag']['time']:.2f} ثانية", f"{results['long']['time']:.2f} ثانية")
        table.add_row("حجم السياق المرسل", "Top-K Segments", "Full Document Text")

        console.print(table)

        # عرض الإجابات في صناديق
        console.print(Panel(results['rag']['ans'], title="إجابة RAG (Nvidia Nemotron)", border_style="green"))
        console.print(Panel(results['long']['ans'], title="إجابة Long Context (Gemini Flash)", border_style="blue"))

# تشغيل النظام
if __name__ == "__main__":
    main()